# College Admissions and Tuition Analysis

## Project Overview

This project analyzes data from 777 U.S. colleges and universities. It combines data cleaning, exploratory data analysis, feature engineering, visualization, correlation analysis, and hypothesis testing to investigate institutional selectivity, tuition, enrollment, and faculty credentials.

## Research Questions

1. Are private universities generally more expensive than public universities?
2. Are larger universities more expensive than smaller universities?
3. Is university size strongly related to out-of-state tuition?
4. Is acceptance rate related to the percentage of students from the top 10% of their high-school class?
5. Are universities with a higher percentage of PhD-qualified faculty more selective?
6. Does higher tuition reduce the number of applications?

## Skills Demonstrated

- Data loading and validation with pandas
- Data cleaning and column standardization
- Exploratory data analysis
- Feature engineering
- Matplotlib and Seaborn visualization
- Correlation analysis
- Pearson hypothesis testing
- Interpretation of statistical results

## Dataset

The notebook expects a file named `College.csv`. The dataset contains 777 institutions and includes admissions, enrollment, tuition, faculty, expenditure, alumni, and graduation variables.


> **Required file:** This notebook needs the original `College.csv` dataset.  
> In VS Code or local Jupyter, place `College.csv` in the same folder as this notebook.  
> In Google Colab, the notebook will ask you to upload the file automatically.

## 1. Import Libraries


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

%matplotlib inline

pd.set_option("display.max_columns", None)


## 2. Load the Dataset


In [ ]:
from pathlib import Path
import io

def load_college_data():
    """
    Load College.csv in either Google Colab or a local Jupyter/VS Code environment.

    Colab:
        If the file is not already present, an upload dialog will appear.

    Local Jupyter / VS Code:
        Place College.csv in the same folder as this notebook.
    """
    candidate_paths = [
        Path("College.csv"),
        Path("./College.csv"),
        Path("/content/College.csv"),
        Path("/content/sample_data/College.csv"),
    ]

    for path in candidate_paths:
        if path.exists():
            print(f"Loading dataset from: {path.resolve()}")
            return pd.read_csv(path)

    # Google Colab upload fallback
    try:
        from google.colab import files
        print("College.csv was not found. Please upload it now.")
        uploaded = files.upload()

        if "College.csv" not in uploaded:
            uploaded_names = ", ".join(uploaded.keys()) or "no files"
            raise FileNotFoundError(
                f"Expected a file named College.csv, but received: {uploaded_names}"
            )

        return pd.read_csv(io.BytesIO(uploaded["College.csv"]))

    except ImportError:
        raise FileNotFoundError(
            "College.csv was not found.\n\n"
            "For VS Code or local Jupyter:\n"
            "1. Put College.csv in the same folder as this notebook.\n"
            "2. Reopen the notebook from that folder.\n"
            "3. Run all cells again.\n\n"
            "For Google Colab, upload College.csv when prompted."
        )

college = load_college_data()
print(f"Loaded {college.shape[0]} rows and {college.shape[1]} columns.")
college.head()

## 3. Data Cleaning and Preparation


In [ ]:
# Use college names as the index when the Names column is available.
if "Names" in college.columns:
    college = college.set_index("Names", drop=False)

# Standardize column names for easier analysis.
college = college.rename(
    columns={
        "Grad.Rate": "grad_rate",
        "S.F.Ratio": "student_faculty_ratio",
        "perc.alumni": "alumni_donation_rate",
        "Room.Board": "room_board",
        "F.Undergrad": "full_time_undergrad",
        "P.Undergrad": "part_time_undergrad",
    }
)

college.columns = (
    college.columns
    .str.strip()
    .str.lower()
    .str.replace(".", "_", regex=False)
)

# Fill missing application values with zero, following the original project specification.
if "apps" in college.columns:
    college["apps"] = college["apps"].fillna(0)

print("Duplicate rows:", college.duplicated().sum())
college.info()


### Data Quality Summary


In [ ]:
quality_summary = pd.DataFrame({
    "dtype": college.dtypes.astype(str),
    "missing_values": college.isna().sum(),
    "unique_values": college.nunique(dropna=False),
})
quality_summary


## 4. Basic Data Questions


In [ ]:
least_tuition_college = college["outstate"].idxmin()
least_tuition_value = college.loc[least_tuition_college, "outstate"]

print("College with the lowest out-of-state tuition:")
print(f"{least_tuition_college}: ${least_tuition_value:,.0f}")


In [ ]:
penn_colleges = college[
    college["names"].str.contains("Penn", case=True, na=False)
]

print(f"Number of institutions containing 'Penn': {len(penn_colleges)}")
penn_colleges[["names", "private", "outstate"]]


## 5. Exploratory Data Analysis


### 5.1 Relationships Among Admissions and Enrollment Variables


In [ ]:
eda_columns = [
    "apps", "accept", "enroll", "top10perc", "top25perc",
    "full_time_undergrad", "part_time_undergrad", "outstate", "room_board"
]

sns.pairplot(college[eda_columns].dropna())
plt.show()


**Interpretation:** Applications, acceptances, and enrollments show strong positive relationships. The two academic-quality measures, `top10perc` and `top25perc`, are also strongly related. Several size-related variables are right-skewed, indicating that a relatively small number of institutions are much larger than the rest.


### 5.2 Private vs. Public Tuition


In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=college, x="private", y="outstate")
plt.title("Out-of-State Tuition by Institution Type")
plt.xlabel("Private Institution")
plt.ylabel("Out-of-State Tuition")
plt.show()

college.groupby("private")["outstate"].agg(["count", "mean", "median"])


**Interpretation:** Private institutions generally have higher median and average out-of-state tuition than public institutions.


## 6. Feature Engineering: University Size


In [ ]:
average_enrollment = college["enroll"].mean()

college["large_university"] = np.where(
    college["enroll"] > average_enrollment,
    "Large",
    "Small"
)

print(f"Average enrollment threshold: {average_enrollment:,.1f}")
college["large_university"].value_counts()


In [ ]:
large_universities = college[college["large_university"] == "Large"]
large_universities.describe(include="all")


### 6.1 Does University Size Affect Tuition?


In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=college, x="large_university", y="outstate")
plt.title("Out-of-State Tuition by University Size")
plt.xlabel("University Size")
plt.ylabel("Out-of-State Tuition")
plt.show()

college.groupby("large_university")["outstate"].agg(["count", "mean", "median"])


**Interpretation:** Larger universities are not necessarily more expensive. In this dataset, smaller institutions tend to have higher typical out-of-state tuition.


### 6.2 Enrollment and Tuition Relationship


In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=college, x="enroll", y="outstate", alpha=0.65)
plt.title("Enrollment vs. Out-of-State Tuition")
plt.xlabel("New Students Enrolled")
plt.ylabel("Out-of-State Tuition")
plt.show()

enroll_tuition_corr = college[["enroll", "outstate"]].corr().iloc[0, 1]
print(f"Pearson correlation: {enroll_tuition_corr:.3f}")


**Interpretation:** The relationship between enrollment size and out-of-state tuition is weak, so enrollment alone is not a strong predictor of tuition.


## 7. Distribution of Applications


In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(college["apps"], bins=50, edgecolor="black")
plt.title("Distribution of College Applications")
plt.xlabel("Applications Received")
plt.ylabel("Number of Institutions")
plt.show()

under_10k = college["apps"].between(0, 10_000, inclusive="left").sum()
between_10k_20k = college["apps"].between(10_000, 20_000, inclusive="left").sum()

print(f"Institutions receiving 0–9,999 applications: {under_10k}")
print(f"Institutions receiving 10,000–19,999 applications: {between_10k_20k}")


**Interpretation:** Most institutions receive fewer than 10,000 applications. The distribution is strongly right-skewed because only a small number of universities receive very large application volumes.


## 8. Acceptance Rate and Student Academic Profile


In [ ]:
college_analysis = college.copy()

college_analysis["acceptance_rate"] = np.where(
    college_analysis["apps"] > 0,
    college_analysis["accept"] / college_analysis["apps"],
    np.nan
)

plt.figure(figsize=(8, 5))
sns.regplot(
    data=college_analysis,
    x="acceptance_rate",
    y="top10perc",
    scatter_kws={"alpha": 0.55},
    ci=None,
)
plt.title("Acceptance Rate vs. Students from Top 10% of High-School Class")
plt.xlabel("Acceptance Rate")
plt.ylabel("Students from Top 10% (%)")
plt.show()

acceptance_top10_corr = (
    college_analysis[["acceptance_rate", "top10perc"]]
    .dropna()
    .corr()
    .iloc[0, 1]
)
print(f"Pearson correlation: {acceptance_top10_corr:.3f}")


**Interpretation:** Acceptance rate has a moderate negative relationship with the percentage of students from the top 10% of their high-school class. More selective institutions tend to enroll academically stronger incoming classes.


## 9. Hypothesis Test 1: PhD Faculty and Selectivity


### Hypotheses

- **H₀:** There is no statistically significant relationship between the percentage of faculty with PhDs and acceptance rate.
- **H₁:** A higher percentage of PhD-qualified faculty is associated with a lower acceptance rate.


In [ ]:
rq1_data = college_analysis[["phd", "acceptance_rate"]].dropna()

r1, p1 = stats.pearsonr(
    rq1_data["phd"],
    rq1_data["acceptance_rate"]
)

plt.figure(figsize=(9, 5))
sns.regplot(
    data=rq1_data,
    x="phd",
    y="acceptance_rate",
    scatter_kws={"alpha": 0.55},
    ci=None,
)
plt.title("Acceptance Rate vs. Percentage of Faculty with PhDs")
plt.xlabel("Faculty with PhDs (%)")
plt.ylabel("Acceptance Rate")
plt.show()

print(f"Correlation: {r1:.4f}")
print(f"P-value: {p1:.4e}")


### Interpretation

The original analysis found a statistically significant negative correlation. A small p-value supports rejecting the null hypothesis, indicating that universities with higher proportions of PhD-qualified faculty tend to have lower acceptance rates.

This is an association, not proof that faculty credentials directly cause greater selectivity.


## 10. Hypothesis Test 2: Tuition and Application Volume


### Hypotheses

- **H₀:** Out-of-state tuition is not related to the number of applications.
- **H₁:** Higher out-of-state tuition is associated with fewer applications.


In [ ]:
rq2_data = college_analysis[["outstate", "apps"]].dropna()

r2, p2 = stats.pearsonr(
    rq2_data["outstate"],
    rq2_data["apps"]
)

plt.figure(figsize=(9, 5))
sns.regplot(
    data=rq2_data,
    x="outstate",
    y="apps",
    scatter_kws={"alpha": 0.30},
    ci=None,
)
plt.title("Out-of-State Tuition vs. Number of Applications")
plt.xlabel("Out-of-State Tuition")
plt.ylabel("Applications Received")
plt.show()

print(f"Correlation: {r2:.4f}")
print(f"P-value: {p2:.4f}")


### Interpretation

The original project reported a p-value above 0.05, so the null hypothesis was not rejected. There was no statistically significant evidence that higher tuition was associated with fewer applications in this dataset.

Application volume is likely influenced by many additional factors, including reputation, location, program offerings, financial aid, and selectivity.


## 11. Key Findings


- Private institutions generally have higher out-of-state tuition than public institutions.
- University size has only a weak relationship with tuition.
- Most colleges receive fewer than 10,000 applications.
- Acceptance rate is negatively associated with the percentage of students from the top 10% of their high-school class.
- A higher percentage of PhD-qualified faculty is associated with lower acceptance rates.
- Tuition alone does not significantly explain application volume in this dataset.


## 12. Limitations and Future Improvements


### Limitations

- The analysis is observational and does not establish causation.
- Pearson correlation captures only linear relationships.
- Institutional data may come from a single historical period.
- Potential confounding variables are not controlled.

### Future Improvements

- Build multivariate regression models for acceptance rate and application volume.
- Examine public and private institutions separately.
- Use robust statistical methods for skewed variables and outliers.
- Add interactive Plotly visualizations.
- Create a dashboard summarizing tuition, enrollment, and selectivity.


## Conclusion

This project demonstrates a complete exploratory analytics workflow, from data preparation and feature engineering through visualization and hypothesis testing. The results show that selectivity is more closely associated with academic characteristics than with tuition or institutional size.
